In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from patsy import dmatrix
from statsmodels.regression.quantile_regression import QuantReg
import warnings
import os
from statsmodels.tools.sm_exceptions import IterationLimitWarning

warnings.simplefilter('ignore', IterationLimitWarning)

# Create directory for saving plots if it doesn't exist
if not os.path.exists('Stiction_Plots'):
    os.makedirs('Stiction_Plots')

# --- 1. Simulation & Fitting Functions ---
def simulate_stiction(input_signal, deadband=0.8):
    output = np.zeros_like(input_signal)
    current_pos = input_signal[0]
    for i in range(len(input_signal)):
        if np.abs(input_signal[i] - current_pos) > deadband:
            current_pos = input_signal[i]
        output[i] = current_pos
    return output

def fit_bspline(x, y, df=40):
    num_internal_knots = df - 3 - 1
    knots = np.percentile(x, np.linspace(0, 100, num_internal_knots + 2)[1:-1])
    X_mat = dmatrix(f"bs(x, knots={list(knots)}, degree=3, include_intercept=False)", {"x": x})
    coeffs = np.linalg.lstsq(X_mat, y, rcond=1e-6)[0]
    return np.array(X_mat @ coeffs).flatten()

def fit_l1_spline(x, y, df=40): 
    num_internal_knots = df - 3 - 1
    knots = np.percentile(x, np.linspace(0, 100, num_internal_knots + 2)[1:-1])
    X_mat = dmatrix(f"bs(x, knots={list(knots)}, degree=3, include_intercept=True)", {"x": x})
    model = QuantReg(y, X_mat)
    try:
        return model.fit(q=0.5, max_iter=2000).predict(X_mat)
    except:
        return model.fit(q=0.5, method='bfgs', max_iter=2000).predict(X_mat)

def get_stats_str(matrix):
    return (f"Mean: {np.mean(matrix):.4f}\nMedian: {np.median(matrix):.4f}\n"
            f"Std: {np.std(matrix):.4f}\nVar: {np.var(matrix):.4f}")

# --- 2. Data Generation & Bootstrapping ---
np.random.seed(42)
X = np.linspace(0, 10, 1000)
y_ic_raw = 5 * np.sin(X) + np.random.normal(0, 0.2, 1000)
y_ooc_raw = simulate_stiction(y_ic_raw, deadband=0.8)

B = 200
print(f"Executing full analysis with {B} bootstraps...")

# Fits
yhat_ic_bs = fit_bspline(X, y_ic_raw)
yhat_ooc_bs = fit_bspline(X, y_ooc_raw)
yhat_ic_l1 = fit_l1_spline(X, y_ic_raw)
yhat_ooc_l1 = fit_l1_spline(X, y_ooc_raw)

# Residuals
res_ic_bs, res_ooc_bs = y_ic_raw - yhat_ic_bs, y_ooc_raw - yhat_ooc_bs
res_ic_l1, res_ooc_l1 = y_ic_raw - yhat_ic_l1, y_ooc_raw - yhat_ooc_l1

# Bootstraps
boot_ic_bs = np.array([fit_bspline(X, yhat_ic_bs + np.random.choice(res_ic_bs, 1000)) for _ in range(B)])
boot_ooc_bs = np.array([fit_bspline(X, yhat_ooc_bs + np.random.choice(res_ooc_bs, 1000)) for _ in range(B)])
boot_ic_l1 = np.array([fit_l1_spline(X, yhat_ic_l1 + np.random.choice(res_ic_l1, 1000)) for _ in range(B)])
boot_ooc_l1 = np.array([fit_l1_spline(X, yhat_ooc_l1 + np.random.choice(res_ooc_l1, 1000)) for _ in range(B)])

# --- 3. Monitoring Metrics ---
ic_meds_bs = np.median(boot_ic_bs, axis=1)
ref_center_bs = np.median(ic_meds_bs)
mad_bs = np.median(np.abs(ic_meds_bs - ref_center_bs))

ic_meds_l1 = np.median(boot_ic_l1, axis=1)
ref_center_l1 = np.median(ic_meds_l1)
mad_l1 = np.median(np.abs(ic_meds_l1 - ref_center_l1))

ooc_meds_bs, ooc_meds_l1 = np.median(boot_ooc_bs, axis=1), np.median(boot_ooc_l1, axis=1)

# Method 1: Pandolfo
depth_ic = 1 / (1 + np.abs(ic_meds_bs - ref_center_bs))
depth_ooc = 1 / (1 + np.abs(ooc_meds_bs - ref_center_bs))
LCL_pan = np.percentile(depth_ic, 5)

# Method 2: Wei Std
d_ic_bs = np.abs(ic_meds_bs - ref_center_bs) / mad_bs
d_ooc_bs = np.abs(ooc_meds_bs - ref_center_bs) / mad_bs
UCL_wei_bs = np.percentile(d_ic_bs, 95)

# Method 3: Wei Robust
d_ic_l1 = np.abs(ic_meds_l1 - ref_center_l1) / mad_l1
d_ooc_l1 = np.abs(ooc_meds_l1 - ref_center_l1) / mad_l1
UCL_wei_l1 = np.percentile(d_ic_l1, 95)

# --- 4. Plotting ---

# FIG 1: Phase I Control Charts
fig1, (ax1a, ax1b, ax1c) = plt.subplots(3, 1, figsize=(10, 11))
ax1a.scatter(range(B), depth_ic, color='blue', alpha=0.5, label='IC $D_p$ values')
ax1a.axhline(LCL_pan, color='green', ls='--', label=f'IC LCL:{LCL_pan:.4f}')
ax1a.set_title("Depth")
ax1a.set_xlabel("Simulation")
ax1a.set_ylabel("$D_p$")
ax1a.legend(loc='lower right', fontsize=20)

ax1b.scatter(range(B), d_ic_bs, color='blue', alpha=0.5, label='IC $D$ values')
ax1b.axhline(UCL_wei_bs, color='red', ls='--', label=f'IC UCL: {UCL_wei_bs:.4f}')
ax1b.set_title("Location (B-spline)")
ax1b.set_xlabel("Simulation")
ax1b.set_ylabel("$D$")
ax1b.legend(loc='lower right', fontsize=20)

ax1c.scatter(range(B), d_ic_l1, color='blue', alpha=0.5, label='IC $D$ values')
ax1c.axhline(UCL_wei_l1, color='red', ls='--', label=f'IC UCL: {UCL_wei_l1:.3f}')
ax1c.set_title("Location ($L_1$ Spline Smoothing)")
ax1c.set_xlabel("Simulation")
ax1c.set_ylabel("$D$")
ax1c.legend(loc='lower right', fontsize=20)
fig1.tight_layout()
fig1.savefig("Stiction_Plots/Stiction_Phase_I_control_charts.png")

# FIG 2: Phase II Control Charts
fig2, (ax2a, ax2b, ax2c) = plt.subplots(3, 1, figsize=(10, 11))
ax2a.scatter(range(B), depth_ooc, color='purple', alpha=0.5, label='OOC $D_p$ values')
ax2a.axhline(LCL_pan, color='red', ls='--', label=f'IC LCL: {LCL_pan:.3f}')
ax2a.set_title("Depth")
ax2a.set_xlabel("Simulation")
ax2a.set_ylabel("$D_p$")
ax2a.legend(loc='lower right', fontsize=20)

ax2b.scatter(range(B), d_ooc_bs, color='purple', alpha=0.5, label='OOC $D$ values')
ax2b.axhline(UCL_wei_bs, color='red', ls='--', label=f'IC UCL: {UCL_wei_bs:.3f}')
ax2b.set_title("Location (B-spline)")
ax2b.set_xlabel("Simulation")
ax2b.set_ylabel("$D$")
ax2b.legend(loc='lower right', fontsize=20)

ax2c.scatter(range(B), d_ooc_l1, color='purple', alpha=0.5, label='OOC $D$ values')
ax2c.axhline(UCL_wei_l1, color='red', ls='--', label=f'IC UCL: {UCL_wei_l1:.3f}')
ax2c.set_title("Location ($L_1$ Spline Smoothing)")
ax2c.set_xlabel("Simulation")
ax2c.set_ylabel("$D$")
ax2c.legend(loc='lower right', fontsize=20)
fig2.tight_layout()
fig2.savefig("Stiction_Plots/Stiction_Phase_II_control_charts.png")

# FIG 3: IC Comparison (1x2)
fig3, (ax3a, ax3b) = plt.subplots(1, 2, figsize=(16, 6))
ax3a.plot(X, y_ic_raw, color='lightgray', alpha=0.9, label='Raw IC')
ax3a.plot(X, yhat_ic_bs, color='blue', lw=4, label='Initial Fit')
ax3a.plot(X, np.mean(boot_ic_bs, axis=0), color='black', ls='--', lw=3, label='Mean Bootstrap Spline Fit')
ax3a.set_title("IC: B-Spline")
ax3a.set_xlabel("X-axis")
ax3a.set_ylabel("Y-axis")
# Increased stat text size to 14
ax3a.text(0.05, 0.05, get_stats_str(boot_ic_bs), transform=ax3a.transAxes, fontsize=16, bbox=dict(facecolor='white', alpha=0.8))
ax3a.legend(loc='lower right', fontsize=14)

ax3b.plot(X, y_ic_raw, color='lightgray', alpha=0.9, label='Raw IC')
ax3b.plot(X, yhat_ic_l1, color='purple', lw=4, label='Initial Fit')
ax3b.plot(X, np.mean(boot_ic_l1, axis=0), color='black', ls='--', lw=3, label='Mean Bootstrap Spline Fit')
ax3b.set_title("IC: $L_1$ Spline Smoothing")
ax3b.set_xlabel("X-axis")
ax3b.set_ylabel("Y-axis")
# Increased stat text size to 14
ax3b.text(0.05, 0.05, get_stats_str(boot_ic_l1), transform=ax3b.transAxes, fontsize=16, bbox=dict(facecolor='white', alpha=0.8))
ax3b.legend(loc='lower right', fontsize=14)
fig3.tight_layout()
fig3.savefig("Stiction_Plots/Stiction_IC_splines.png")

# FIG 4: OOC Comparison (1x2)
fig4, (ax4a, ax4b) = plt.subplots(1, 2, figsize=(16, 6))
ax4a.plot(X, y_ooc_raw, color='wheat', alpha=0.9, label='Raw OOC')
ax4a.plot(X, yhat_ooc_bs, color='red', lw=4, label='Initial Fit')
ax4a.plot(X, np.mean(boot_ooc_bs, axis=0), color='black', ls='--', lw=3, label='Mean Bootstrap Spline Fit')
ax4a.set_title("OOC: B-Spline")
ax4a.set_xlabel("X-axis")
ax4a.set_ylabel("Y-axis")
ax4a.text(0.05, 0.05, get_stats_str(boot_ooc_bs), transform=ax4a.transAxes, fontsize=16, bbox=dict(facecolor='white', alpha=0.8))
ax4a.legend(loc='lower right', fontsize=14)

ax4b.plot(X, y_ooc_raw, color='wheat', alpha=0.9, label='Raw OOC')
ax4b.plot(X, yhat_ooc_l1, color='brown', lw=4, label='Initial Fit')
ax4b.plot(X, np.mean(boot_ooc_l1, axis=0), color='black', ls='--', lw=3, label='Mean Bootstrap Spline Fit')
ax4b.set_title("OOC: $L_1$ Spline Smoothing")
ax4b.set_xlabel("X-axis")
ax4b.set_ylabel("Y-axis")
ax4b.text(0.05, 0.05, get_stats_str(boot_ooc_l1), transform=ax4b.transAxes, fontsize=16, bbox=dict(facecolor='white', alpha=0.8))
ax4b.legend(loc='lower right', fontsize=14)
fig4.tight_layout()
fig4.savefig("Stiction_Plots/Stiction_OOC_splines.png")

plt.show()

print(f"\nDetection Summary:\nPandolfo: {np.sum(depth_ooc < LCL_pan)}\nWei Std: {np.sum(d_ooc_bs > UCL_wei_bs)}\nWei Robust: {np.sum(d_ooc_l1 > UCL_wei_l1)}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from patsy import dmatrix
from math import factorial
import os
import warnings


plt.rcParams['legend.fontsize'] = 20

# Create directory for saving plots if it doesn't exist
if not os.path.exists('Stiction_Plots'):
    os.makedirs('Stiction_Plots')

# --- 1. Normalized Permutation Entropy Function ---
def permutation_entropy(signal, d=5, t=1):
    n = len(signal)
    indices = np.arange(d) * t
    embedded_vectors = np.array([signal[i + indices] for i in range(n - (d - 1) * t)])
    permutations = [tuple(np.argsort(v)) for v in embedded_vectors]
    counts = {}
    for p in permutations:
        counts[p] = counts.get(p, 0) + 1
    probs = np.array(list(counts.values())) / len(permutations)
    npe = -np.sum(probs * np.log(probs)) / np.log(factorial(d))
    return npe

# --- 2. Simulation & Fitting Functions ---
def simulate_stiction(input_signal, deadband=0.8):
    output = np.zeros_like(input_signal)
    current_pos = input_signal[0]
    for i in range(len(input_signal)):
        if np.abs(input_signal[i] - current_pos) > deadband:
            current_pos = input_signal[i]
        output[i] = current_pos
    return output

def fit_bspline(x, y, df=40):
    num_internal_knots = df - 3 - 1
    knots = np.percentile(x, np.linspace(0, 100, num_internal_knots + 2)[1:-1])
    X_mat = dmatrix(f"bs(x, knots={list(knots)}, degree=3, include_intercept=False)", {"x": x})
    coeffs = np.linalg.lstsq(X_mat, y, rcond=1e-6)[0]
    return np.array(X_mat @ coeffs).flatten()

def bootstrap_splines(x, y, yhat_initial, B=200):
    residuals = y - yhat_initial
    return np.array([fit_bspline(x, yhat_initial + np.random.choice(residuals, len(y))) for _ in range(B)])

# --- 3. Execution ---
np.random.seed(42)
X = np.linspace(0, 10, 1000)
y_ic_raw = 5 * np.sin(X) + np.random.normal(0, 0.2, 1000)
y_ooc_raw = simulate_stiction(y_ic_raw, deadband=0.8)

yhat_ic_initial = fit_bspline(X, y_ic_raw)
yhat_ooc_initial = fit_bspline(X, y_ooc_raw)

B = 200
print(f"Generating {B} bootstraps and calculating variants...")
boot_ic = bootstrap_splines(X, y_ic_raw, yhat_ic_initial, B)
boot_ooc = bootstrap_splines(X, y_ooc_raw, yhat_ooc_initial, B)

# --- 4. Plotting ---

# PLOT 1: Signal Comparison
fig1, (ax1_top, ax1_bot) = plt.subplots(2, 1, figsize=(14, 12)) 
ax1_top.plot(X, y_ic_raw, color='lightgray', label='Raw IC')
ax1_top.plot(X, yhat_ic_initial, 'k', lw=2, label='Initial Fit')
ax1_top.plot(X, np.mean(boot_ic, axis=0), 'r--', label='Avg Boot')
ax1_top.set_title("IC Signal Summary")
ax1_top.set_xlabel("X-axis")
ax1_top.set_ylabel("Y-axis")
ax1_top.legend(loc='upper right')

ax1_bot.plot(X, y_ooc_raw, color='wheat', label='Raw OOC')
ax1_bot.plot(X, yhat_ooc_initial, 'k', lw=2, label='Initial Fit')
ax1_bot.plot(X, np.mean(boot_ooc, axis=0), 'r--', label='Avg Boot')
ax1_bot.set_title("OOC Signal Summary")
ax1_bot.set_xlabel("X-axis")
ax1_bot.set_ylabel("Y-axis")
ax1_bot.legend(loc='upper right')
plt.tight_layout()
plt.savefig('Stiction_Plots/Signal_comparison.png')

# PLOTS 2 & 3: Distributions
for phase, data, fig_num in [('IC', boot_ic, 2), ('OOC', boot_ooc, 3)]:
    fig, (ax_mean, ax_med) = plt.subplots(2, 1, figsize=(10, 7))
    ax_mean.hist(np.mean(data, axis=1), bins=25, color='skyblue', edgecolor='black')
    ax_mean.set_title(f"Distribution of Bootstrap Means ({phase})")
    ax_mean.set_xlabel("Mean Value")
    ax_mean.set_ylabel("Frequency")
    
    ax_med.hist(np.median(data, axis=1), bins=25, color='salmon', edgecolor='black')
    ax_med.set_title(f"Distribution of Bootstrap Medians ({phase})")
    ax_med.set_xlabel("Median Value")
    ax_med.set_ylabel("Frequency")
    plt.tight_layout()
    plt.savefig(f'Stiction_Plots/Distributions_{phase}.png')

# PLOTS 4 & 5: NPE Control Charts
d_vals, t_vals = [3, 4, 5], [1, 2]
detection_results = []

for phase, boot_data, fig_num in [('IC', boot_ic, 4), ('OOC', boot_ooc, 5)]:
    fig, axs = plt.subplots(3, 2, figsize=(10, 11.5), sharex=True) 
    
    for r, d in enumerate(d_vals):
        for c, t in enumerate(t_vals):
            # Calculate metrics for current sub-plot
            npe_vals = np.array([permutation_entropy(curve, d=d, t=t) for curve in boot_data])
            ic_npe_temp = np.array([permutation_entropy(curve, d=d, t=t) for curve in boot_ic])
            lcl, ucl = np.percentile(ic_npe_temp, [2.5, 97.5])
            
            # Plotting
            axs[r, c].plot(npe_vals, 'o', color='purple', alpha=0.6, markersize=4, label=f'{phase} H-values')
            axs[r, c].axhline(ucl, color='red', ls='--', label=f'UCL: {ucl:.4f}')
            axs[r, c].axhline(lcl, color='green', ls='--', label=f'LCL: {lcl:.4f}')
            
            axs[r, c].set_title(f"d={d}, $\\tau$={t}")
            axs[r, c].set_xlabel("Simulation")
            axs[r, c].set_ylabel("$H$")
            
            # Global legend for every subplot at bottom right
            axs[r, c].legend(loc='lower right', framealpha=0.8)
            
            if phase == 'OOC':
                alarms = np.sum((npe_vals >= ucl) | (npe_vals <= lcl))
                detection_results.append({'d': d, 't': t, 'Alarms': alarms})

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(f'Stiction_Plots/NPE_Control_Charts_{phase}.png')

plt.show()

# --- 5. Summary Stats ---
meds_ic, meds_ooc = np.median(boot_ic, axis=1), np.median(boot_ooc, axis=1)
print(f"\nIC Median   : {np.min(meds_ic):.4f} to {np.max(meds_ic):.4f}")
print(f"OOC Median  : {np.min(meds_ooc):.4f} to {np.max(meds_ooc):.4f}")

# --- 5. Summary Stats ---
mean_ic, mean_ooc = np.mean(boot_ic, axis=1), np.mean(boot_ooc, axis=1)
print(f"\nIC Mean  : {np.min(mean_ic):.4f} to {np.max(mean_ic):.4f}")
print(f"OOC Mean  : {np.min(mean_ooc):.4f} to {np.max(mean_ooc):.4f}")

# Print Alarm Table
print("\n" + "="*30)
print(f"{'d':<5} | {'t':<5} | {'OOC Alarms':<10}")
print("-" * 30)
for res in detection_results:
    print(f"{res['d']:<5} | {res['t']:<5} | {res['Alarms']:<10}")
print("="*30)